# zagg query pipeline

Three AOIs, full ICESat-2 mission timeseries. Catalogs and shard maps land in `outputs/` (consumed by the write notebook).

In [1]:
# %pip install "zagg[catalog,viz]"

import logging
import time
from datetime import date
from pathlib import Path

import numpy as np

from zagg.catalog import load_polygon
from zagg.catalog.shardmap import ShardMap
from zagg.catalog.sources import Catalog, CMRSource, Query, STACQuery, STACSource
from zagg.config import default_config
from zagg.data import demo_aoi
from zagg.grids import HealpixGrid, from_config
from zagg.viz import show_shardmap

In [ ]:
# Exact-S2 intersection backend: spherely fork wheels (SpatialIndex branch) from the
# demo bucket -- macOS arm64 + manylinux x86_64, cp310-cp314. Uses ambient AWS
# credentials (CryoCloud hub role / AWS profile). Without a matching wheel everything
# below still runs: ShardMap.build's backend="auto" falls back to mortie.
import importlib.util

if importlib.util.find_spec("spherely") is None:
    import sys

    try:
        import boto3

        s3 = boto3.client("s3")
        Path("wheels").mkdir(exist_ok=True)
        listing = s3.list_objects_v2(Bucket="sliderule-public", Prefix="zagg-demo/wheels/")
        for obj in listing["Contents"]:
            name = obj["Key"].rsplit("/", 1)[1]
            if name.endswith(".whl"):
                s3.download_file("sliderule-public", obj["Key"], f"wheels/{name}")
        !{sys.executable} -m pip install -q spherely --no-index --find-links wheels/
    except Exception as e:
        print(f"wheel fetch failed ({e}); continuing without spherely")

try:
    import spherely

    print("spherely", spherely.__version__, "| SpatialIndex:", hasattr(spherely, "SpatialIndex"))
except ImportError:
    print("spherely unavailable -- ShardMap.build will use the mortie backend")

In [3]:
## 0 — Timing infra

In [4]:
logging.getLogger("stac_geoparquet").setLevel(logging.WARNING)

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
MISSION = ("2018-10-13", date.today().isoformat())

# The one machine-local input: example 2's full ATL03 catalog clone (305 MB,
# not packaged). Point this at your clone.
ATL03_CLONE = Path.home() / "software/zagg/data/atl03_v007/atl03_v007_full.parquet"

timings = {}

class stage:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        timings[self.name] = round(time.perf_counter() - self.t0, 2)
        print(f"[{self.name}] {timings[self.name]:.1f}s")


def healpix_grid(order):
    # ~10 m leaf cells (child order 19); footprint-intersection MOC at order 13 (#92)
    return HealpixGrid(order, child_order=19, chunk_inner=13)

## 1 — NEON SERC site

### 1a — NASA CMR: ATL03

In [5]:
serc = demo_aoi("serc")
grid9 = healpix_grid(9)
fetch_bbox = grid9.coverage_bbox(serc)  # shard-complete: cover the whole shards, not just the AOI

with stage("SERC: CMR query"):
    cat_serc = CMRSource().fetch(Query("ATL03", "007", *MISSION, region=serc))#fetch_bbox))
cat_serc.to_geoparquet(str(OUT / "catalog_atl03_serc.parquet"))
print(f"{len(cat_serc):,} granules")

[SERC: CMR query] 2.7s
68 granules


In [6]:
with stage("SERC: shardmap build (o9)"):
    sm_serc = ShardMap.build(cat_serc, grid9, region=load_polygon(serc), mortie_order=9)
sm_serc.to_json(str(OUT / "shardmap_atl03_serc_o9.json"))
{k: sm_serc.metadata[k] for k in ("total_shards", "total_granules", "total_pairs", "build_wall_s")}

[SERC: shardmap build (o9)] 0.0s


{'total_shards': 4,
 'total_granules': 68,
 'total_pairs': 213,
 'build_wall_s': 0.008}

In [7]:
show_shardmap(sm_serc, cat_serc, aoi=serc, zoom=10)

Map(center=[38.87384105897101, -76.552734375], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoo…

### 1b — STAC (Earth Search): Sentinel-2 L2A

In [8]:
s2_source = STACSource(
    "https://earth-search.aws.element84.com/v1",
    assets=["red", "green", "blue", "nir", "scl"],
    time_key="s2:datatake_id",
)
s2_query = STACQuery(
    collections=["sentinel-2-c1-l2a", "sentinel-2-pre-c1-l2a"],
    start_date="2015-06-23",
    end_date=MISSION[1],
    region=fetch_bbox,
)
with stage("SERC: STAC query"):
    cat_s2 = s2_source.fetch(s2_query)
cat_s2.to_geoparquet(str(OUT / "catalog_s2_serc.parquet"))
datatakes = {r["time_key"] for r in cat_s2.granule_records()}
print(f"{len(cat_s2):,} items, {len(datatakes):,} datatakes")

[SERC: STAC query] 4.3s
1,059 items, 526 datatakes


In [9]:
s2_config = default_config("sentinel2_l2a")
s2_config.output["grid"]["parent_order"] = 9

with stage("SERC: S2 shardmap build (o9)"):
    sm_s2 = ShardMap.build(cat_s2, from_config(s2_config), region=load_polygon(serc), mortie_order=9)
sm_s2.to_json(str(OUT / "shardmap_s2_serc_o9.json"))
{k: sm_s2.metadata[k] for k in ("total_shards", "total_granules", "total_pairs", "build_wall_s")}

[SERC: S2 shardmap build (o9)] 0.1s


{'total_shards': 4,
 'total_granules': 1059,
 'total_pairs': 3167,
 'build_wall_s': 0.014}

In [10]:
show_shardmap(sm_s2, cat_s2, aoi=serc, zoom=9)

Map(center=[38.87384105897101, -76.552734375], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoo…

## 2 — California, full mission: CMR vs a local catalog clone

Same CMR pipeline as SERC at state scale — then the identical cut against a local clone of the full ATL03 catalog, which is why the clone exists.

In [11]:
with stage("load ATL03 catalog clone"):
    full = Catalog.from_geoparquet(str(ATL03_CLONE))
print(f"{len(full):,} granules (full mission, global)")

[NEON: load local catalog clone] 0.5s
555,867 granules (full mission, global)


In [ ]:
ca = demo_aoi("california")
fetch_bbox_ca = healpix_grid(9).coverage_bbox(ca)

with stage("California: CMR query"):
    cat_ca_cmr = CMRSource().fetch(Query("ATL03", "007", *MISSION, region=fetch_bbox_ca))

with stage("California: local catalog cut"):
    cat_ca = full.filter_bbox([fetch_bbox_ca])

print(
    f"CMR: {len(cat_ca_cmr):,} granules in {timings['California: CMR query']:.0f}s | "
    f"local clone: {len(cat_ca):,} granules in {timings['California: local catalog cut']:.2f}s"
)
cat_ca.to_geoparquet(str(OUT / "catalog_atl03_california.parquet"))

In [16]:
ca_parts = load_polygon(ca)
with stage("California: shardmap build (o9)"):
    sm_ca9 = ShardMap.build(cat_ca, healpix_grid(9), region=ca_parts, mortie_order=9)
sm_ca9.to_json(str(OUT / "shardmap_california_o9.json"))
{k: sm_ca9.metadata[k] for k in ("total_shards", "total_granules", "total_pairs", "build_wall_s")}

[California: shardmap build (o9)] 6.5s


{'total_shards': 2721,
 'total_granules': 2686,
 'total_pairs': 195574,
 'build_wall_s': 6.334}

Reprocess o9 → o8 in place: HEALPix nesting makes coarsening a pure regroup — no re-query, no geometry. (Caveat: shard-completeness is order-relative — o8 edge shards are wider than the o9 fetch box, so refetch with `healpix_grid(8).coverage_bbox(ca)` if the coarser store must stay shard-complete.)

In [ ]:
with stage("California: reproject o9 -> o8"):
    sm_ca8 = sm_ca9.reproject(healpix_grid(8))
sm_ca8.to_json(str(OUT / "shardmap_california_o8.json"))
{**sm_ca8.metadata["reproject"], "total_shards": sm_ca8.metadata["total_shards"], "total_pairs": sm_ca8.metadata["total_pairs"]}

In [ ]:
show_shardmap(sm_ca9, aoi=ca, zoom=6)

In [ ]:
show_shardmap(sm_ca8, aoi=ca, zoom=6)

## 3 — All NEON AOP domains (D01–D20)

61 flight-box parts, Alaska to Puerto Rico — the union bbox spans most of North America, so this is the local-clone path only: one shard-complete box per part.

In [12]:
neon_parts = load_polygon(demo_aoi("neon_aop"))
boxes = [healpix_grid(9).coverage_bbox([p]) for p in neon_parts]  # per part: one box per flight box

with stage("NEON: bbox prefilter"):
    cat_neon = full.filter_bbox(boxes)
print(f"{len(cat_neon):,} candidate granules over {len(neon_parts)} flight-box parts")

[NEON: bbox prefilter] 0.2s
22,116 candidate granules over 61 flight-box parts


In [13]:
with stage("NEON: shardmap build (o9)"):
    sm_neon = ShardMap.build(cat_neon, healpix_grid(9), region=neon_parts, mortie_order=9)
sm_neon.to_json(str(OUT / "shardmap_neon_aop_o9.json"))
{k: sm_neon.metadata[k] for k in ("total_shards", "total_granules", "total_pairs", "build_wall_s")}

[NEON: shardmap build (o9)] 2.6s


{'total_shards': 250,
 'total_granules': 22116,
 'total_pairs': 19999,
 'build_wall_s': 1.659}

In [14]:
show_shardmap(sm_neon, aoi=demo_aoi("neon_aop"), zoom=4)

Map(center=[44.686324250206326, -111.8839039522059], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Timings

In [ ]:
import json

import pandas as pd

(OUT / "timings_query.json").write_text(json.dumps(timings, indent=2))
pd.Series(timings, name="seconds").to_frame()